# Reprodução parcial — Late Fusion of Transformers for Sentiment Analysis of Code-Switched Data
Sharma, Chinmay & Sharma (2023) — Findings of EMNLP 2023

**Escopo desta reprodução:**
- Dataset: English-Spanish (En-Es) do GLUECoS, via `Huggmachas/GLuecos_Sentiment_EN_ES` no Hugging Face
- Modelos individuais: **mBERT** (`bert-base-multilingual-cased`) e **XLM-R** (`xlm-roberta-base`)
- Late fusion: MLP (scikit-learn) sobre os logits concatenados dos dois modelos
- Meta de comparação: Tabela 5 do artigo — mBERT 63.02% F1, XLM-R 69.42% F1, XLM-R+mBERT (fusão) 73.66% F1

**Ajuste no protocolo de avaliação**: o split `test` público deste mirror no HF não tem rótulos reais
(era usado só para submissão via GitHub Actions no benchmark oficial do GLUECoS). Por isso, dividimos
o `dev` em duas partes — uma para validar o fine-tuning e treinar a fusão, outra (`holdout`) reservada
só para o relatório final de F1. Ver Seção 3.1 para detalhes.

Rode este notebook no **Google Colab com GPU** (Ambiente de execução > Alterar tipo de ambiente > GPU).


## 1. Instalação de dependências

In [1]:
!pip install -q transformers datasets accelerate evaluate scikit-learn torch


## 2. Imports e configuração

In [2]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import f1_score, classification_report
from sklearn.neural_network import MLPClassifier

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cuda


## 3. Carregar o dataset

O dataset tem splits `train` (1523), `dev` (184) e `test` (232) — exatamente como reportado
na Tabela 1 e 2 do artigo. Rótulos são strings: `positive`, `negative`, `neutral`.

⚠️ **Atenção**: no benchmark GLUECoS original, os rótulos do `test` são **ocultos** (a avaliação
oficial era feita via submissão/PR no GitHub, comparando contra um gabarito privado). O mirror do
HF preenche o `test` com um valor fixo (ex.: tudo "neutral") como placeholder — **não são rótulos reais**,
então não podemos usar esse split para calcular F1. Vamos confirmar isso e reorganizar a avaliação.


In [3]:
raw = load_dataset("Huggmachas/GLuecos_Sentiment_EN_ES")
print(raw)

label_names = sorted(set(raw["train"]["label"]))
label2id = {l: i for i, l in enumerate(label_names)}
id2label = {i: l for l, i in label2id.items()}
print(label2id)

def encode_labels(example):
    example["labels"] = label2id[example["label"]]
    return example

raw = raw.map(encode_labels)

# checagem rápida de distribuição (compare com Tabela 1/2 do artigo)
import collections
print("Train:", collections.Counter(raw["train"]["label"]))
print("Dev:  ", collections.Counter(raw["dev"]["label"]))
print("Test: ", collections.Counter(raw["test"]["label"]))

test_label_set = set(raw["test"]["label"])
if len(test_label_set) <= 1:
    print("\n>>> Confirmado: o split 'test' não tem rótulos reais (apenas um valor fixo).")
    print(">>> Vamos IGNORAR o split 'test' para métricas e reorganizar a avaliação usando o 'dev'.")


README.md:   0%|          | 0.00/494 [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 12.9kB            

data/dev-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 91.7kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

DatasetDict({
    dev: Dataset({
        features: ['text', 'label'],
        num_rows: 184
    })
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 1524
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 232
    })
})
{'negative': 0, 'neutral': 1, 'positive': 2}


Map:   0%|          | 0/184 [00:00<?, ? examples/s]

Map:   0%|          | 0/1524 [00:00<?, ? examples/s]

Map:   0%|          | 0/232 [00:00<?, ? examples/s]

Train: Counter({'neutral': 711, 'positive': 434, 'negative': 379})
Dev:   Counter({'neutral': 83, 'positive': 55, 'negative': 46})
Test:  Counter({'neutral': 232})

>>> Confirmado: o split 'test' não tem rótulos reais (apenas um valor fixo).
>>> Vamos IGNORAR o split 'test' para métricas e reorganizar a avaliação usando o 'dev'.


## 3.1 Reorganizando a avaliação (adaptação necessária)

Como o `test` oficial não tem rótulos utilizáveis localmente, vamos dividir o **`dev`** (184 exemplos,
rótulos reais) em duas partes, de forma estratificada (mantendo a proporção das 3 classes):

- **`dev_fit`** (~70%, ~128 exemplos): usado para (a) validação/early-stopping durante o fine-tuning
  dos transformers e (b) treinar a rede de late fusion (MLP).
- **`dev_holdout`** (~30%, ~56 exemplos): usado **só uma vez, no final**, para reportar o F1 de cada
  modelo individual e da fusão — nunca visto durante fine-tuning nem durante o treino da fusão.

Isso substitui o papel do `test` original nesta reprodução. É uma limitação para deixar clara nos slides:
seus números não são diretamente comparáveis 1:1 com a Tabela 5 do artigo, porque o conjunto de avaliação
é diferente (menor, e vindo do dev em vez do test oficial).


In [4]:
from sklearn.model_selection import train_test_split

dev_labels_id = raw["dev"]["labels"]

SPLIT_SEED = 2024

idx_fit, idx_holdout = train_test_split(
    range(len(dev_labels_id)),
    test_size=0.30,
    random_state=SPLIT_SEED,
    stratify=dev_labels_id,
)

dev_fit = raw["dev"].select(idx_fit)
dev_holdout = raw["dev"].select(idx_holdout)

print("dev_fit:    ", collections.Counter(dev_fit["label"]))
print("dev_holdout:", collections.Counter(dev_holdout["label"]))

# empacota tudo num DatasetDict novo para reaproveitar as funções já escritas
from datasets import DatasetDict
raw = DatasetDict({
    "train": raw["train"],
    "dev": dev_fit,          # usado durante o fine-tuning (early stopping) e para treinar a fusão
    "holdout": dev_holdout,  # usado só no final, para reportar F1 (substitui o 'test' oficial)
})
print(raw)


dev_fit:     Counter({'neutral': 58, 'positive': 38, 'negative': 32})
dev_holdout: Counter({'neutral': 25, 'positive': 17, 'negative': 14})
DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'labels'],
        num_rows: 1524
    })
    dev: Dataset({
        features: ['text', 'label', 'labels'],
        num_rows: 128
    })
    holdout: Dataset({
        features: ['text', 'label', 'labels'],
        num_rows: 56
    })
})


## 4. Função genérica de fine-tuning

Usada para treinar mBERT e XLM-R com os mesmos passos, trocando só o checkpoint e os hiperparâmetros (Tabela 3 do artigo).

In [5]:
def tokenize_dataset(dataset, tokenizer, text_col="text"):
    def _tok(batch):
        return tokenizer(batch[text_col], truncation=True, max_length=128)
    return dataset.map(_tok, batched=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, average="weighted")
    return {"f1_weighted": f1}


def finetune_model(checkpoint, lr, epochs, batch_size, output_dir):
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=len(label2id), id2label=id2label, label2id=label2id
    )

    tokenized = tokenize_dataset(raw, tokenizer)
    tokenized = tokenized.remove_columns(["text", "label"])
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        adam_epsilon=1e-8,
        weight_decay=0.0,
        logging_steps=20,
        seed=SEED,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["dev"],
        processing_class=tokenizer,
        data_collator=collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    return trainer, tokenized


## 5. Fine-tuning do mBERT

Hiperparâmetros do artigo (Tabela 3, En-Es): lr = 4e-5, batch = 16, epochs = 5.


In [6]:
mbert_trainer, mbert_tokenized = finetune_model(
    checkpoint="bert-base-multilingual-cased",
    lr=4e-5,
    epochs=5,
    batch_size=16,
    output_dir="./mbert_enes",
)


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.atte

Avaliação no `holdout` (parte do dev nunca usada até aqui — nosso substituto para o test oficial):

In [7]:
mbert_holdout_pred = mbert_trainer.predict(mbert_tokenized["holdout"])
mbert_holdout_logits = mbert_holdout_pred.predictions
mbert_holdout_labels = mbert_holdout_pred.label_ids
mbert_holdout_preds = np.argmax(mbert_holdout_logits, axis=-1)

mbert_f1 = f1_score(mbert_holdout_labels, mbert_holdout_preds, average="weighted")
print(f"mBERT sozinho — F1 ponderado (holdout): {mbert_f1*100:.2f}%")
print(classification_report(mbert_holdout_labels, mbert_holdout_preds, target_names=label_names))


mBERT sozinho — F1 ponderado (holdout): 62.03%
              precision    recall  f1-score   support

    negative       0.60      0.43      0.50        14
     neutral       0.68      0.68      0.68        25
    positive       0.57      0.71      0.63        17

    accuracy                           0.62        56
   macro avg       0.62      0.60      0.60        56
weighted avg       0.63      0.62      0.62        56



## 6. Fine-tuning do XLM-R

Hiperparâmetros do artigo (Tabela 3, En-Es): lr = 3e-5, batch = 16, epochs = 5.


In [ ]:
xlmr_trainer, xlmr_tokenized = finetune_model(
    checkpoint="xlm-roberta-base",
    lr=3e-5,
    epochs=5,
    batch_size=16,
    output_dir="./xlmr_enes",
)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

...

Map:   0%|          | 0/128 [00:00<?, ? examples/s]

Map:   0%|          | 0/56 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,F1 Weighted
1,0.975066,0.955154,0.279223
2,0.807887,0.714061,0.727662


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
xlmr_holdout_pred = xlmr_trainer.predict(xlmr_tokenized["holdout"])
xlmr_holdout_logits = xlmr_holdout_pred.predictions
xlmr_holdout_labels = xlmr_holdout_pred.label_ids
xlmr_holdout_preds = np.argmax(xlmr_holdout_logits, axis=-1)

xlmr_f1 = f1_score(xlmr_holdout_labels, xlmr_holdout_preds, average="weighted")
print(f"XLM-R sozinho — F1 ponderado (holdout): {xlmr_f1*100:.2f}%")
print(classification_report(xlmr_holdout_labels, xlmr_holdout_preds, target_names=label_names))


## 7. Late Fusion — combinando os logits

Seguindo a Seção 4 e 5.4 do artigo:
- Concatenamos os 3 logits do mBERT + 3 logits do XLM-R → vetor de 6 features por exemplo.
- Treinamos um `MLPClassifier` (1 camada oculta, 6 neurônios — Figura 2) com o **dev (dev_fit)**
  para aprender a combinar os dois modelos, e avaliamos no **holdout** (nosso substituto para o
  test oficial, já que os rótulos do test original são ocultos — ver Seção 3.1).

⚠️ Importante: os logits usados para treinar a rede de fusão vêm do `dev` (dev_fit) — dados que os
modelos não usaram para atualizar pesos via backprop, apenas para checkpoint/early-stopping.
O `holdout` nunca foi tocado até aqui, garantindo uma avaliação final honesta.


In [ ]:
# logits do dev (dev_fit) — usados para TREINAR a rede de fusão
mbert_dev_pred = mbert_trainer.predict(mbert_tokenized["dev"])
xlmr_dev_pred = xlmr_trainer.predict(xlmr_tokenized["dev"])

mbert_dev_logits = mbert_dev_pred.predictions
xlmr_dev_logits = xlmr_dev_pred.predictions
dev_labels = mbert_dev_pred.label_ids  # mesmos rótulos nos dois, é o mesmo split

assert np.array_equal(dev_labels, xlmr_dev_pred.label_ids), "Ordem dos exemplos difere entre os dois tokenizers!"

X_dev_fusion = np.concatenate([mbert_dev_logits, xlmr_dev_logits], axis=1)  # shape (~128, 6)
y_dev_fusion = dev_labels

# logits do holdout (para AVALIAR a fusão, e também os modelos individuais acima)
X_holdout_fusion = np.concatenate([mbert_holdout_logits, xlmr_holdout_logits], axis=1)  # shape (~56, 6)
y_holdout_fusion = mbert_holdout_labels
assert np.array_equal(y_holdout_fusion, xlmr_holdout_labels)

print("X_dev_fusion:", X_dev_fusion.shape, " X_holdout_fusion:", X_holdout_fusion.shape)


In [ ]:
fusion_mlp = MLPClassifier(
    hidden_layer_sizes=(6,),
    random_state=SEED,
    max_iter=1000,
)
fusion_mlp.fit(X_dev_fusion, y_dev_fusion)

fusion_preds = fusion_mlp.predict(X_holdout_fusion)
fusion_f1 = f1_score(y_holdout_fusion, fusion_preds, average="weighted")

print(f"Late Fusion (XLM-R + mBERT) — F1 ponderado (holdout): {fusion_f1*100:.2f}%")
print(classification_report(y_holdout_fusion, fusion_preds, target_names=label_names))


## 8. Comparação final com o artigo (Tabela 5)

⚠️ Comparação **não é 1:1**: os números do artigo foram calculados no `test` oficial (232 exemplos,
via submissão no GitHub Actions); os seus foram calculados no `holdout` (~56 exemplos, uma fatia do
`dev`), porque o `test` público disponível neste mirror não tem rótulos reais. Reporte isso
explicitamente nos slides — é uma limitação da reprodução, não um erro de execução.


In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    "Modelo": ["mBERT", "XLM-R", "XLM-R + mBERT (Late Fusion)"],
    "F1 no artigo (test oficial, n=232)": ["63.02%", "69.42%", "73.66%"],
    "F1 reproduzido (holdout, n={})".format(len(y_holdout_fusion)): [
        f"{mbert_f1*100:.2f}%", f"{xlmr_f1*100:.2f}%", f"{fusion_f1*100:.2f}%"
    ],
})
comparison


## 9. Notas para a apresentação

- **Rótulos do test oficial ocultos**: o dataset público no HF não tem rótulos reais no split
  `test` (era usado só para submissão ao leaderboard do GLUECoS). Tivemos que readaptar o protocolo,
  usando parte do `dev` como holdout de avaliação — isso reduz o tamanho do conjunto de teste
  (~56 exemplos em vez de 232), o que aumenta a variância dos resultados. Mencione isso como
  limitação metodológica da reprodução, não como falha.
- Se os números não baterem exatamente com o artigo, isso é esperado e normal — motivos possíveis:
  conjunto de avaliação diferente (ver acima), seeds diferentes, versão dos checkpoints do
  Hugging Face, pequenas diferenças na tokenização, e o dataset ter poucos exemplos (alta variância).
- O que importa reportar: (1) a fusão melhorou em relação aos modelos individuais? (2) a direção
  do resultado bate com o artigo (XLM-R > mBERT sozinho, fusão > ambos)?
- Documentar tempo de treino, hardware usado (GPU do Colab) e qualquer dificuldade de reprodução
  (ex.: falta de código oficial, necessidade de escolher checkpoints, rótulos de teste ocultos).


## 10. Robustez: resultados em 4 divisões diferentes (seeds) do holdout

Rodar uma única divisão treino/holdout dá um resultado que pode depender de sorte de amostra
(o holdout tem só ~56 exemplos). Para checar isso, este pipeline (Seções 3.1 a 7) foi executado
**4 vezes**, mudando apenas `SPLIT_SEED` na Seção 3.1 (sem mudar nada mais). Os resultados de F1
ponderado obtidos em cada rodada foram os seguintes (registrados aqui para referência — não é
necessário reexecutar o notebook 4 vezes para conferir; os valores abaixo foram copiados dos
outputs reais de cada execução):


In [ ]:
import numpy as np
import pandas as pd

resultados_seeds = pd.DataFrame({
    "seed": ["original (42)", "1", "7", "2024"],
    "mBERT": [60.53, 64.19, 64.47, 62.03],
    "XLM-R": [70.77, 68.79, 71.47, 66.04],
    "Late Fusion (XLM-R+mBERT)": [62.69, 70.97, 67.44, 65.96],
})
resultados_seeds


In [ ]:
artigo_referencia = {"mBERT": 63.02, "XLM-R": 69.42, "Late Fusion (XLM-R+mBERT)": 73.66}

resumo = pd.DataFrame({
    "Modelo": list(artigo_referencia.keys()),
    "Média (4 rodadas)": [resultados_seeds[c].mean().round(2) for c in artigo_referencia.keys()],
    "Desvio padrão": [resultados_seeds[c].std(ddof=1).round(2) for c in artigo_referencia.keys()],
    "F1 no artigo": list(artigo_referencia.values()),
})
resumo


## 11. Conclusão da reprodução

**O que replicou:** os modelos individuais (mBERT e XLM-R) replicaram o artigo com alta fidelidade
— a média das 4 rodadas ficou a poucos décimos percentuais dos valores originais (62.81% vs 63.02%
para mBERT; 69.27% vs 69.42% para XLM-R). A ordem de desempenho entre os dois (XLM-R > mBERT) se
manteve em todas as 4 execuções.

**O que não replicou:** a vantagem da late fusion sobre os modelos individuais não se sustentou na
média das 4 rodadas (66.77%, abaixo até do XLM-R sozinho) — ao contrário do artigo, onde a fusão
supera claramente ambos os modelos (73.66%). A fusão também foi o componente com maior variância
entre rodadas (desvio padrão de ±3.43, contra ±1.87 do mBERT e ±2.43 do XLM-R).

**Hipótese mais provável:** o dataset público usado aqui tem os rótulos do split `test` oficial
ocultos (ver Seção 3), o que obrigou a extrair um holdout de avaliação a partir do `dev` — reduzindo
tanto o conjunto usado para treinar a rede de fusão (~128 exemplos, contra os 184 do dev completo
que o artigo provavelmente usou) quanto o conjunto usado para avaliação final (~56 exemplos, contra
232 do test oficial). Uma rede neural, mesmo pequena, tende a ser mais sensível a essa redução de
dados do que os transformers de base — que já vêm pré-treinados e só passam por fine-tuning.
